# 함수형 변환 · apply · 파이프라인 설계 — CSV → 정제 → 인코딩/스케일링 → 저장 파이프라인

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.preprocessing import RobustScaler

## 0. 원본 CSV 준비

In [2]:
np.random.seed(42)
n = 1500
price = np.random.choice([9900, 19900, 29900, 49900, 89900, 249900], n)
quantity = np.random.randint(1, 4, n)
raw = pd.DataFrame({
    "order_id": [f"O{i:05d}" for i in range(1, n + 1)],
    "customer_age": np.random.randint(18, 70, n).astype(float),
    "region": np.random.choice(["서울", " 서울 ", "Seoul", "경기", "부산", "대구", "인천"], n),
    "membership": np.random.choice(["basic", "silver", "gold", "vip", "VIP"], n, p=[.5, .25, .15, .07, .03]),
    "channel": np.random.choice(["web", "app", "app ", "APP"], n, p=[.45, .45, .05, .05]),
    "category": np.random.choice(["가전", "패션", "식품", "뷰티", "생활"], n),
    "price": price.astype(float),
    "quantity": quantity,
    "amount": (price * quantity).astype(float),
})
raw.loc[np.random.choice(n, 40, replace=False), "amount"] = np.nan        # 결측
raw.loc[np.random.choice(n, 30, replace=False), "customer_age"] = np.nan  # 결측
raw.loc[np.random.choice(n, 10, replace=False), "customer_age"] = 999     # 이상치
raw.loc[np.random.choice(n, 5, replace=False), "quantity"] = 80           # 이상치

work = Path("d006_work"); work.mkdir(exist_ok=True)
input_csv = work / "orders_raw.csv"
output_csv = work / "orders_clean.csv"
raw.to_csv(input_csv, index=False)
print("원본 CSV 저장:", input_csv, raw.shape)

원본 CSV 저장: d006_work/orders_raw.csv (1500, 9)


## 1. 단계 함수 (한 함수 = 한 책임)

In [3]:
def load_orders(path):
    # CSV에서 주문 데이터를 읽어 옵니다.
    return pd.read_csv(path)

def clean_strings_full(df):
    # 문자열 컬럼의 공백·대소문자·표기 혼재를 정리합니다.
    return df.assign(
        region=df["region"].str.strip().replace({"Seoul": "서울"}),
        membership=df["membership"].str.lower(),
        channel=df["channel"].str.strip().str.lower(),
    )

def drop_invalid_rows(df, age_min=10, age_max=80, qty_max=10):
    # 불가능한 값과 결측을 제거합니다.
    return (
        df
        .dropna(subset=["amount", "customer_age"])
        .query("@age_min < customer_age < @age_max")
        .query("quantity <= @qty_max")
        .drop_duplicates()
        .reset_index(drop=True)
    )

def add_derived(df):
    # 분석용 파생 컬럼을 추가합니다.
    return df.assign(
        amount_log=lambda d: np.log1p(d["amount"]),
        is_premium=lambda d: d["membership"].isin(["gold", "vip"]).astype(int),
        amount_class=lambda d: np.where(d["amount"] >= 100_000, "high",
                                np.where(d["amount"] >= 30_000, "mid", "low"))
    )

def encode_full(df):
    # membership=Ordinal, 나머지=One-Hot.
    order_map = {"basic": 1, "silver": 2, "gold": 3, "vip": 4}
    out = df.assign(membership_ord=df["membership"].map(order_map))
    one_hots = [
        pd.get_dummies(out["region"],   prefix="region", dtype=int),
        pd.get_dummies(out["channel"],  prefix="ch",     dtype=int),
        pd.get_dummies(out["category"], prefix="cat",    dtype=int),
    ]
    return pd.concat([out] + one_hots, axis=1)

def scale_with_robust(df, cols=("customer_age", "amount", "quantity")):
    # 수치형 컬럼을 Robust로 스케일링하고 _scaled 접미사를 붙입니다.
    scaler = RobustScaler()
    scaled = scaler.fit_transform(df[list(cols)])
    scaled_df = pd.DataFrame(scaled,
                             columns=[f"{c}_scaled" for c in cols],
                             index=df.index)
    return pd.concat([df, scaled_df], axis=1)

## 2. 파이프라인 조립 — 한 줄 호출

In [4]:
def preprocess(input_path):
    # 원본 CSV 경로를 받아 전처리된 DataFrame을 돌려줍니다.
    return (
        load_orders(input_path)
        .pipe(clean_strings_full)
        .pipe(drop_invalid_rows, age_min=10, age_max=80, qty_max=10)
        .pipe(add_derived)
        .pipe(encode_full)
        .pipe(scale_with_robust)
    )

clean_df = preprocess(input_csv)
print("출력 shape:", clean_df.shape)
clean_df.head(3)

출력 shape: (1415, 28)


,order_id,customer_age,region,membership,channel,category,price,quantity,amount,amount_log,...,ch_app,ch_web,cat_가전,cat_뷰티,cat_생활,cat_식품,cat_패션,customer_age_scaled,amount_scaled,quantity_scaled
0,O00001,37.0,부산,vip,app,패션,49900.0,2,99800.0,11.510933,...,1,0,0,0,0,0,1,-0.269231,0.266845,0.0
1,O00002,41.0,인천,basic,web,생활,89900.0,2,179800.0,12.099606,...,0,1,0,0,1,0,0,-0.115385,0.800534,0.0
2,O00003,42.0,서울,basic,web,패션,29900.0,3,89700.0,11.404237,...,0,1,0,0,0,0,1,-0.076923,0.199466,0.5


## 3. 저장 + 재로드 검증

In [5]:
clean_df.to_csv(output_csv, index=False)

reloaded = pd.read_csv(output_csv)
print("저장 파일:", output_csv, "—", output_csv.stat().st_size, "bytes")
print("저장 직전 shape:", clean_df.shape)
print("다시 읽은  shape:", reloaded.shape)
print("동일한가?:", clean_df.shape == reloaded.shape)

저장 파일: d006_work/orders_clean.csv — 208385 bytes
저장 직전 shape: (1415, 28)
다시 읽은  shape: (1415, 28)
동일한가?: True


## 4. 품질 리포트

In [6]:
def quality_report(before: pd.DataFrame, after: pd.DataFrame) -> dict:
    # 전처리 전·후 데이터의 품질 차이를 dict로 반환합니다.
    return {
        "rows_before": before.shape[0],
        "rows_after":  after.shape[0],
        "removed_pct": round(100 * (1 - after.shape[0] / before.shape[0]), 2),
        "cols_before": before.shape[1],
        "cols_after":  after.shape[1],
        "new_cols":    after.shape[1] - before.shape[1],
        "missing_top5_before": before.isnull().sum().sort_values(ascending=False).head(5).to_dict(),
        "missing_top5_after":  after.isnull().sum().sort_values(ascending=False).head(5).to_dict(),
        "dtypes_after": after.dtypes.astype(str).value_counts().to_dict(),
    }

rep = quality_report(load_orders(input_csv), clean_df)
for k, v in rep.items():
    print(f"{k}: {v}")

rows_before: 1500
rows_after: 1415
removed_pct: 5.67
cols_before: 9
cols_after: 28
new_cols: 19
missing_top5_before: {'amount': 40, 'customer_age': 30, 'order_id': 0, 'membership': 0, 'region': 0}
missing_top5_after: {'order_id': 0, 'customer_age': 0, 'region': 0, 'membership': 0, 'channel': 0}
dtypes_after: {'int64': 15, 'float64': 7, 'str': 6}
